In [1]:
# bibliotecas necessárias

# biblioteca: pandas
#!pip install pandas
import pandas as pd

# bibliotecas: io (imput-output binário para string)
import io

# biblioteca: plotly
#!pip install plotly
import plotly.express as px
import plotly
import plotly.graph_objects as go

# biblioteca: numpy
#!pip install numpy
import numpy as np

# biblioteca: folium (mapas)
#!pip install folium
import folium

# biblioteca: haversine (mapas)
#!pip install haversine
from haversine import haversine

#biblioteca: streamlit
#%pip install streamlit
import streamlit as st

In [2]:
df = pd.read_csv (r'dataset/train.csv')

df.head()

,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min)
0,0x4607,INDORES13DEL02,37,4.9,22.745049,75.892471,22.765049,75.912471,19-03-2022,11:30:00,11:45:00,conditions Sunny,High,2,Snack,motorcycle,0,No,Urban,(min) 24
1,0xb379,BANGRES18DEL02,34,4.5,12.913041,77.683237,13.043041,77.813237,25-03-2022,19:45:00,19:50:00,conditions Stormy,Jam,2,Snack,scooter,1,No,Metropolitian,(min) 33
2,0x5d6d,BANGRES19DEL01,23,4.4,12.914264,77.678400,12.924264,77.688400,19-03-2022,08:30:00,08:45:00,conditions Sandstorms,Low,0,Drinks,motorcycle,1,No,Urban,(min) 26
3,0x7a6a,COIMBRES13DEL02,38,4.7,11.003669,76.976494,11.053669,77.026494,05-04-2022,18:00:00,18:10:00,conditions Sunny,Medium,0,Buffet,motorcycle,1,No,Metropolitian,(min) 21
4,0x70a2,CHENRES12DEL01,32,4.6,12.972793,80.249982,13.012793,80.289982,26-03-2022,13:30:00,13:45:00,conditions Cloudy,High,1,Snack,scooter,1,No,Metropolitian,(min) 30


In [3]:
df1 = df.copy()

#Selecionando apenas linhas filtradas em um campo específico e salvando como cópia retirando "NaN "
linhas_selecionadas = df1['Delivery_person_Age'] !='NaN '
df1 = df1.loc[linhas_selecionadas,:].copy()

linhas_selecionadas = df1['Road_traffic_density'] !='NaN '
df1 = df1.loc[linhas_selecionadas,:].copy()

linhas_selecionadas = df1['City'] !='NaN '
df1 = df1.loc[linhas_selecionadas,:].copy()

linhas_selecionadas = df1['Festival'] !='NaN '
df1 = df1.loc[linhas_selecionadas,:].copy()

linhas_selecionadas = df1['Order_Date'] !='NaN'
df1 = df1.loc[linhas_selecionadas,:].copy()

#Convertendo dados tipo número inteiro
df1 ['Delivery_person_Age'] = df1 ['Delivery_person_Age'].astype (int)

#Convertendo dados tipo número decimal
df1 ['Delivery_person_Ratings'] = df1 ['Delivery_person_Ratings'].astype (float)

#Convertendo dados tipo texto em data
df1['Order_Date'] = pd.to_datetime(df1['Order_Date'],format='%d-%m-%Y')

#Selecionando apenas linhas filtradas em um campo específico e salvando como cópia retirando "NaN " e convertando para inteiro
linhas_selecionadas = df1['multiple_deliveries'] !='NaN '
df1 = df1.loc[linhas_selecionadas,:].copy()
df1 ['multiple_deliveries'] = df1 ['multiple_deliveries'].astype (int)

#transforma em string e retira todas os espaços após a string.
df1.loc[:, 'ID'] = df1.loc[:, 'ID'].str.strip()
df1.loc[:, 'Road_traffic_density'] = df1.loc[:, 'Road_traffic_density'].str.strip()
df1.loc[:, 'Type_of_order'] = df1.loc[:, 'Type_of_order'].str.strip()
df1.loc[:, 'Type_of_vehicle'] = df1.loc[:, 'Type_of_vehicle'].str.strip()
df1.loc[:, 'City'] = df1.loc[:, 'City'].str.strip()
df1.loc[:, 'Festival'] = df1.loc[:, 'Festival'].str.strip()

# Limpeza da coluna de tempo
df1['Time_taken(min)'] = df1['Time_taken(min)'].apply(lambda x: x.split('(min)')[1])
df1['Time_taken(min)'] = df1['Time_taken(min)'].astype (int)

In [4]:
df1.head()

,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min)
0,0x4607,INDORES13DEL02,37,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,11:30:00,11:45:00,conditions Sunny,High,2,Snack,motorcycle,0,No,Urban,24
1,0xb379,BANGRES18DEL02,34,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,19:45:00,19:50:00,conditions Stormy,Jam,2,Snack,scooter,1,No,Metropolitian,33
2,0x5d6d,BANGRES19DEL01,23,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,08:30:00,08:45:00,conditions Sandstorms,Low,0,Drinks,motorcycle,1,No,Urban,26
3,0x7a6a,COIMBRES13DEL02,38,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,18:00:00,18:10:00,conditions Sunny,Medium,0,Buffet,motorcycle,1,No,Metropolitian,21
4,0x70a2,CHENRES12DEL01,32,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,13:30:00,13:45:00,conditions Cloudy,High,1,Snack,scooter,1,No,Metropolitian,30


In [5]:
df1.dtypes

ID                                        str
Delivery_person_ID                        str
Delivery_person_Age                     int64
Delivery_person_Ratings               float64
Restaurant_latitude                   float64
Restaurant_longitude                  float64
Delivery_location_latitude            float64
Delivery_location_longitude           float64
Order_Date                     datetime64[us]
Time_Orderd                               str
Time_Order_picked                         str
Weatherconditions                         str
Road_traffic_density                      str
Vehicle_condition                       int64
Type_of_order                             str
Type_of_vehicle                           str
multiple_deliveries                     int64
Festival                                  str
City                                      str
Time_taken(min)                         int64
dtype: object

# Visão entregadores

In [6]:
# 1. A menor e maior idade dos entregadores.

idade_min = df1['Delivery_person_Age'].min()
idade_max = df1['Delivery_person_Age'].max()

print('A maior idade é: {}'.format (df1['Delivery_person_Age'].min()))
print('A menor idade é: {}'.format(df1['Delivery_person_Age'].max()))

A maior idade é: 20
A menor idade é: 39


In [7]:
# 2. A pior e a melhor condição de veículos.

idade_min = df1['Vehicle_condition'].min()
idade_max = df1['Vehicle_condition'].max()

print('A melhor condição de veículo é: {}'.format (df1['Vehicle_condition'].min()))
print('A pior condição de veículo é: {}'.format(df1['Vehicle_condition'].max()))

A melhor condição de veículo é: 0
A pior condição de veículo é: 2


In [8]:
# 3. A avaliação média por entregador.

cols = ['Delivery_person_ID', 'Delivery_person_Ratings']
ID_rating_mean = df1.loc[:,cols].groupby('Delivery_person_ID').mean().reset_index()
ID_rating_mean

,Delivery_person_ID,Delivery_person_Ratings
0,AGRRES010DEL01,4.761538
1,AGRRES010DEL02,4.671429
2,AGRRES010DEL03,4.575000
3,AGRRES01DEL01,4.522222
4,AGRRES01DEL02,4.700000
...,...,...
1315,VADRES19DEL02,4.632727
1316,VADRES19DEL03,4.670270
1317,VADRES20DEL01,4.620370
1318,VADRES20DEL02,4.591111


In [9]:
# 4. A avaliação média e o desvio padrão por tipo de tráfego.

cols = ['Road_traffic_density','Delivery_person_Ratings']

# Função de agregação (agg) de funções sobre um campo calculado
avaliacao_med_dp = df1.loc[:,cols].groupby('Road_traffic_density').agg({'Delivery_person_Ratings':['mean','std']})

# mudança do título dos campos
avaliacao_med_dp.columns = ['Delivery mean','Delivery std']

avaliacao_med_dp.reset_index()

,Road_traffic_density,Delivery mean,Delivery std
0,High,4.652230,0.273044
1,Jam,4.594019,0.329778
2,Low,4.645011,0.338080
3,Medium,4.660138,0.274245


In [10]:
# 5. A avaliação média e o desvio padrão por condições climáticas.

cols = ['Weatherconditions','Delivery_person_Ratings']

# Função de agregação (agg) de funções sobre um campo calculado
clima_med_dp = df1.loc[:,cols].groupby('Weatherconditions').agg({'Delivery_person_Ratings':['mean','std']})

# mudança do título dos campos
clima_med_dp.columns = ['Delivery mean','Delivery std']

clima_med_dp.reset_index()

,Weatherconditions,Delivery mean,Delivery std
0,conditions Cloudy,4.651871,0.281197
1,conditions Fog,4.652965,0.275060
2,conditions Sandstorms,4.611748,0.310852
3,conditions Stormy,4.611819,0.313096
4,conditions Sunny,4.654868,0.396674
5,conditions Windy,4.616128,0.304565


In [11]:
# 6. Os 10 entregadores mais rápidos por cidade.

cols = ['Delivery_person_ID', 'Time_taken(min)','City']
df2 = df1.loc[:,cols].groupby(['City','Delivery_person_ID']).min().sort_values(['City','Time_taken(min)'],ascending=True).reset_index()

df_aux01 = df2.loc[df2['City'] == 'Metropolitian',:].head(10)
df_aux02 = df2.loc[df2['City'] == 'Urban',:].head(10)
df_aux03 = df2.loc[df2['City'] == 'Semi-Urban',:].head(10)

df3 = pd.concat([df_aux01,df_aux02,df_aux03]).reset_index(drop=True)

df3

,City,Delivery_person_ID,Time_taken(min)
0,Metropolitian,AGRRES010DEL03,10
1,Metropolitian,AGRRES07DEL03,10
2,Metropolitian,AGRRES12DEL01,10
3,Metropolitian,AGRRES14DEL01,10
4,Metropolitian,AGRRES17DEL03,10
5,Metropolitian,ALHRES08DEL03,10
6,Metropolitian,ALHRES12DEL01,10
7,Metropolitian,ALHRES14DEL02,10
8,Metropolitian,ALHRES15DEL02,10
9,Metropolitian,ALHRES19DEL03,10


In [12]:
# 7. Os 10 entregadores mais lentos por cidade.

cols = ['Delivery_person_ID', 'Time_taken(min)','City']
df2 = df1.loc[:,cols].groupby(['City','Delivery_person_ID']).max().sort_values(['City','Time_taken(min)'],ascending=False).reset_index()

df_aux01 = df2.loc[df2['City'] == 'Metropolitian',:].head(10)
df_aux02 = df2.loc[df2['City'] == 'Urban',:].head(10)
df_aux03 = df2.loc[df2['City'] == 'Semi-Urban',:].head(10)

df3 = pd.concat([df_aux01,df_aux02,df_aux03]).reset_index(drop=True)

df3

,City,Delivery_person_ID,Time_taken(min)
0,Metropolitian,ALHRES010DEL01,54
1,Metropolitian,AURGRES13DEL01,54
2,Metropolitian,BANGRES02DEL01,54
3,Metropolitian,BHPRES16DEL02,54
4,Metropolitian,CHENRES02DEL02,54
5,Metropolitian,CHENRES04DEL01,54
6,Metropolitian,CHENRES07DEL03,54
7,Metropolitian,CHENRES08DEL01,54
8,Metropolitian,CHENRES13DEL02,54
9,Metropolitian,COIMBRES010DEL03,54


# Visão dos restaurantes

In [13]:
df1.dtypes

ID                                        str
Delivery_person_ID                        str
Delivery_person_Age                     int64
Delivery_person_Ratings               float64
Restaurant_latitude                   float64
Restaurant_longitude                  float64
Delivery_location_latitude            float64
Delivery_location_longitude           float64
Order_Date                     datetime64[us]
Time_Orderd                               str
Time_Order_picked                         str
Weatherconditions                         str
Road_traffic_density                      str
Vehicle_condition                       int64
Type_of_order                             str
Type_of_vehicle                           str
multiple_deliveries                     int64
Festival                                  str
City                                      str
Time_taken(min)                         int64
dtype: object

In [14]:
# 1. A quantidade de entregadores únicos.

len(df1['Delivery_person_ID'].unique())

1320

In [15]:
# 2. A distância média dos resturantes e dos locais de entrega.

cols =['Restaurant_latitude', 'Restaurant_longitude', 'Delivery_location_latitude', 'Delivery_location_longitude']

# função "apply" permite realizar o comando "haversine" considerando as variáveis de cada linha, lambda é a declaração da variável independente (x). "axis" identifica se percorre linha(1) ou coluna (0)
df1['distancia'] = df1.loc[:,cols].apply(lambda x:haversine((x['Restaurant_latitude'],x['Restaurant_longitude']),(x['Delivery_location_latitude'],x['Delivery_location_longitude'])),axis=1)
avg_distancia = df1['distancia'].mean()
print('A distância média de entregas e: {:.2f}KM'.format(avg_distancia))

A distância média de entregas e: 27.44KM


In [16]:
# 3. O tempo médio e o desvio padrão de entrega por cidade. agg({'y' : 'calculo de agregação'})

cols = ['City', 'Time_taken(min)']

tempo_med_dp = df1.loc[:,cols].groupby('City').agg({'Time_taken(min)':['mean','std']})

tempo_med_dp.columns = ['tempo_med', 'tempo_dp']
tempo_med_dp = tempo_med_dp.reset_index()
tempo_med_dp

,City,tempo_med,tempo_dp
0,Metropolitian,27.428083,9.133374
1,Semi-Urban,49.710526,2.724992
2,Urban,23.209379,8.858049


In [17]:
# 4. O tempo médio e o desvio padrão de entrega por cidade e tipo de pedido.

cols = ['City', 'Time_taken(min)','Type_of_order']

tempo_med_dp = df1.loc[:,cols].groupby(['City','Type_of_order']).agg({'Time_taken(min)':['mean','std']})

tempo_med_dp.columns = ['tempo_med', 'tempo_dp']
tempo_med_dp = tempo_med_dp.reset_index()
tempo_med_dp

,City,Type_of_order,tempo_med,tempo_dp
0,Metropolitian,Buffet,27.299008,9.153107
1,Metropolitian,Drinks,27.322691,9.041655
2,Metropolitian,Meal,27.616383,9.214536
3,Metropolitian,Snack,27.468414,9.119676
4,Semi-Urban,Buffet,49.707317,2.731702
5,Semi-Urban,Drinks,49.625000,2.459347
6,Semi-Urban,Meal,50.300000,3.041665
7,Semi-Urban,Snack,49.408163,2.707385
8,Urban,Buffet,23.560652,9.056348
9,Urban,Drinks,23.311977,8.927314


In [18]:
# 5. O tempo médio e o desvio padrão de entrega por cidade e tipo de tráfego.

cols = ['City', 'Time_taken(min)','Road_traffic_density']

tempo_med_dp = df1.loc[:,cols].groupby(['City','Road_traffic_density']).agg({'Time_taken(min)':['mean','std']})

tempo_med_dp.columns = ['tempo_med', 'tempo_dp']
tempo_med_dp = tempo_med_dp.reset_index()
tempo_med_dp

,City,Road_traffic_density,tempo_med,tempo_dp
0,Metropolitian,High,28.140898,7.904645
1,Metropolitian,Jam,31.976991,9.476203
2,Metropolitian,Low,22.257675,6.794772
3,Metropolitian,Medium,27.729966,8.308064
4,Semi-Urban,High,50.125000,2.629956
5,Semi-Urban,Jam,49.841270,2.717095
6,Semi-Urban,Medium,47.400000,2.011080
7,Urban,High,24.305335,8.494842
8,Urban,Jam,27.993164,10.078271
9,Urban,Low,19.446809,6.319963


In [19]:
# 6. O tempo médio de entrega durantes os Festivais.

cols = ['Time_taken(min)', 'Festival']

tempo_med_dp = df1.loc[:,cols].groupby(['Festival']).agg({'Time_taken(min)':['mean','std']})

tempo_med_dp.columns = ['tempo_med', 'tempo_dp']
tempo_med_dp = tempo_med_dp.reset_index()
tempo_med_dp

,Festival,tempo_med,tempo_dp
0,No,26.162741,9.001803
1,Yes,45.518607,4.005399


# FUNÇÃO

def x(y):
    valor_de_y = a²+ 2ab + c

    return valor_de_y

# Função Gráfico de Linhas
px.line (tabela_fonte, x = 'variavel_ind' ,y = 'variável_dep' )

In [20]:
# Gráfico de linhas

tabela_fonte = df1.loc[:,['ID','Road_traffic_density']].groupby('Road_traffic_density').count().reset_index()

px.line(tabela_fonte, x = 'Road_traffic_density', y = 'ID')

In [21]:
cols = ['ID','Order_Date']
df1_graf1 = df1.loc[:,cols].groupby(['Order_Date']).count().reset_index()
px.bar(df1_graf1 , x='Order_Date', y='ID')

In [22]:
# Converte em semana do ano (1 a 52) sendo início domingo (U) ou segunda (W)
df1['week_of_year'] = df1['Order_Date'].dt.strftime('%U')
df1.head()

cols = ['ID', 'week_of_year']
df1_graf2 = df1.loc[:,cols].groupby('week_of_year').count().reset_index()
px.line(df1_graf2, x= 'week_of_year', y = 'ID')

In [23]:
df_aux = df1.loc[:,['ID','Road_traffic_density']].groupby('Road_traffic_density').count().reset_index()
df_aux = df_aux.loc[df_aux['Road_traffic_density']!= 'NaN',:]
df_aux['entregas_perc'] = df_aux['ID']/df_aux['ID'].sum()
px.pie(df_aux, values ='entregas_perc', names = 'Road_traffic_density')

In [24]:
cols = (['ID','City','Road_traffic_density'])
agrupamento = (['City','Road_traffic_density'])
df_aux = df1.loc[:,cols].groupby(agrupamento).count().reset_index()
df_aux = df_aux.loc[df_aux['City']!= 'NaN',:]
df_aux = df_aux.loc[df_aux['Road_traffic_density']!= 'NaN',:]
px.scatter(df_aux, x='City', y = 'Road_traffic_density', size='ID', color = 'City')

In [25]:
# Qt. de entregas na semana / número únicode entregadores por semana

df_aux01= df1.loc[:,['ID', 'week_of_year']].groupby('week_of_year').count().reset_index()
df_aux02= df1.loc[:,['Delivery_person_ID', 'week_of_year']].groupby('week_of_year').nunique().reset_index()
df_aux = pd.merge(df_aux01, df_aux02,how = 'inner' )
df_aux [ 'order_by_deliver'] = df_aux['ID']/df_aux['Delivery_person_ID']
px.line(df_aux, x='week_of_year', y = 'order_by_deliver')

In [26]:
cols= (['City','Road_traffic_density','Delivery_location_latitude','Delivery_location_longitude' ])
df_aux = df1.loc[:,cols].groupby(['City','Road_traffic_density']).median().reset_index()
df_aux = df_aux.loc[df_aux['City']!= 'NaN',:]
df_aux = df_aux.loc[df_aux['Road_traffic_density']!= 'NaN',:]
df_aux

map = folium.Map()

for index,location_info in df_aux.iterrows():
  folium.Marker([location_info['Delivery_location_latitude'],location_info['Delivery_location_longitude' ]],
  popup=location_info[['City','Road_traffic_density']]).add_to(map)
map

In [27]:
df1.info()

<class 'pandas.DataFrame'>
Index: 41419 entries, 0 to 45592
Data columns (total 22 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   ID                           41419 non-null  str           
 1   Delivery_person_ID           41419 non-null  str           
 2   Delivery_person_Age          41419 non-null  int64         
 3   Delivery_person_Ratings      41368 non-null  float64       
 4   Restaurant_latitude          41419 non-null  float64       
 5   Restaurant_longitude         41419 non-null  float64       
 6   Delivery_location_latitude   41419 non-null  float64       
 7   Delivery_location_longitude  41419 non-null  float64       
 8   Order_Date                   41419 non-null  datetime64[us]
 9   Time_Orderd                  41419 non-null  str           
 10  Time_Order_picked            41419 non-null  str           
 11  Weatherconditions            41419 non-null  str         

In [28]:
df1.describe()

,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Vehicle_condition,multiple_deliveries,Time_taken(min),distancia
count,41419.000000,41368.000000,41419.000000,41419.000000,41419.000000,41419.000000,41419,41419.000000,41419.000000,41419.000000,41419.000000
mean,29.609382,4.633209,17.250381,70.768890,17.473196,70.832773,2022-03-13 15:14:13.434414,0.995920,0.748183,26.552017,27.439304
min,20.000000,2.500000,-30.902872,0.000000,0.010000,0.010000,2022-02-11 00:00:00,0.000000,0.000000,10.000000,1.465069
25%,25.000000,4.500000,12.933298,73.170283,12.989096,73.279083,2022-03-04 00:00:00,0.000000,0.000000,19.000000,4.663607
50%,30.000000,4.700000,18.554382,75.898497,18.636947,76.002574,2022-03-15 00:00:00,1.000000,1.000000,26.000000,9.271901
75%,35.000000,4.900000,22.732225,78.046106,22.785536,78.107636,2022-03-27 00:00:00,2.000000,1.000000,33.000000,13.736144
max,39.000000,5.000000,30.914057,88.433452,31.054057,88.563452,2022-04-06 00:00:00,2.000000,3.000000,54.000000,6884.735909
std,5.764918,0.315861,7.705868,21.140027,7.341144,21.140228,NaN,0.817978,0.572849,9.333189,303.204975
